# 03 – Skill extraction

Goal: turn free-text job descriptions into a clean list of skills per posting.

1. Build a skill dictionary (canonical names + aliases).
2. Extract skills with the dictionary (MVP).
3. Evaluate against gsearch `description_tokens` (precision / recall).

In [1]:
import ast
import pandas as pd

pd.set_option("display.max_rows", 200)

gs = pd.read_csv("../data/raw/gsearch/gsearch_jobs.csv")
gs = gs.drop_duplicates(["title", "company_name", "location"]).copy()

gs["tokens"] = gs["description_tokens"].fillna("[]").apply(ast.literal_eval)
print("postings:", len(gs))
print("postings without any skill:", round((gs["tokens"].str.len() == 0).mean(), 3))
gs["tokens"].str.len().describe().round(1)

postings: 41719
postings without any skill: 0.23


count    41719.0
mean         3.2
std          3.0
min          0.0
25%          1.0
50%          3.0
75%          5.0
max         25.0
Name: tokens, dtype: float64

In [2]:
vocab = gs["tokens"].explode().value_counts()
print("unique skills:", len(vocab))
vocab

unique skills: 133


tokens
sql                      20065
excel                    12882
python                   12320
power_bi                 11006
tableau                  10799
r                         7352
sas                       3365
powerpoint                2962
word                      2921
azure                     2596
aws                       2476
snowflake                 2411
spark                     2056
looker                    1759
go                        1587
spreadsheet               1513
sap                       1441
scala                     1328
spss                      1174
jira                      1168
c                         1142
java                      1135
hadoop                    1052
alteryx                   1049
outlook                    977
nosql                      925
sharepoint                 901
javascript                 883
vba                        877
gcp                        847
bigquery                   792
redshift                   778
q

In [3]:
import re

def show_context(skill, pattern, n=6, width=60):
    """Print n random snippets where the skill's pattern appears."""
    sub = gs[gs["tokens"].apply(lambda t: skill in t)]
    for d in sub["description"].sample(n, random_state=3):
        m = re.search(pattern, d, flags=re.IGNORECASE)
        if m:
            print("...", d[max(0, m.start() - width): m.end() + width].replace("\n", " "), "...")

show_context("go", r"\bgo\b")

... m. Paper resumes will not be accepted. All job seekers must go to the web site to be considered for positions. If you are  ...
... years or over.  Working on this project will require you to go through a standard recruitment process (including passing a ...
... is position.  Please note: Candidate(s) will be required to go through pre-employment screening.  Unissant, Inc. is a prou ...
... ctional teams  Features and Benefits while On Contract:  We go beyond the basic staffing agency offerings You can see the  ...
... ust apply for open positions through their Workday account. Go to Careers app on WD home screen>Click on 'Find Jobs-Intern ...
... Revenue Operations Remote - USA ... The HubSpot Go To Market team uses data from a wide variety of sources to  ...


In [4]:
import re

def show_context(skill, pattern, n=6, width=60):
    """Print n random snippets where the skill's pattern appears."""
    sub = gs[gs["tokens"].apply(lambda t: skill in t)]
    for d in sub["description"].sample(n, random_state=3):
        m = re.search(pattern, d, flags=re.IGNORECASE)
        if m:
            print("...", d[max(0, m.start() - width): m.end() + width].replace("\n", " "), "...")

show_context("c", r"\bc\b")

... perty Intelligence & Risk Management Platform leading the P&C insurance companies depend on to identify and mitigate risk ...
... logy Supervisor II - 365007 $93,396 - $125,136 Annual Range C... View Job ShareThis Copy and Paste #J-18808-Ljbffr ...
... experience, especially data-focused systems • Proficient in C# and Powershell • Good orientation in Azure services • Know ...
... , we trust our associates to innovate the future of s upply c hain & transportation !  As a Transportation Analyst , you' ...
... is, data mapping, and data modeling • Experience with .Net, C#, Aspx, or Javascript  Compensation: • Hourly W2 pay rate $ ...
... f a Quantitative methods worksheet with 3 sections A, B and C to be competed includes questions that needs to be answered ...


### gsearch skill vocabulary

- 133 skills, median 3 per posting, 23% of postings have none.
- It is a silver standard, not gold: "go" (1,587) is almost always the English verb, "c" (1,142) mostly noise (P&C, "A, B and C", C#).
- Duplicate names: postgres/postgresql, mongo/mongodb, js/javascript, node/node.js ...
- Missing: modern data / ML / GenAI tools (dbt, databricks, kafka, mlflow, xgboost, langchain, hugging face ...) and concepts (machine learning, statistics, nlp, a/b testing).